In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pickle as pkl
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix as cm

In [2]:
df_main = pd.read_csv('../data/narrative_craft.csv')
df_main

,IncidentId,Incident Date,Narrative,Craft Type,ACC Group,ACC,ACC Subtype,LSAR Craft Type
0,1014,2017-01-05 11:05:00.000,The station mechanic was asked by harbour staf...,Fishing Vessel,Commercial,Commercial fishing,Injured/ill,Vessel
1,1020,2017-01-21 22:00:00.000,Launched on service to a broken down dinghy to...,Powered Boat,Powered boats,Motorboat,Machinery/equipment failure,Vessel
2,1037,2017-01-10 15:52:00.000,Was called by a member of the public to a repo...,Rowing Boat,Manual watercraft,Rowing boat,Capsize/swamping,Small Craft
3,1040,2017-01-12 14:00:00.000,WHILST COMING BACK DOWN MORTLAKE REACH WE NOTI...,Other Powered Vessel,Powered boats,Motorboat,Unknown craft,Vessel
4,1048,2017-01-04 11:14:00.000,Launched to 10M fishing vessel Likely Lad WH32...,Fishing Vessel,Commercial,Commercial fishing,Machinery/equipment failure,Vessel
...,...,...,...,...,...,...,...,...
19661,368779,2022-07-25 10:57:00.000,"13 Metre sailing boat, 1.4 metre draught. has ...",Yacht (with engine),Sailing,Sailing vessel,Machinery/equipment failure,Vessel
19662,368786,2022-04-30 21:46:00.000,Launched to a report of a boat broken down nea...,Other Powered Vessel,Powered boats,Motorboat,Machinery/equipment failure,Vessel
19663,368788,2022-08-04 18:15:00.000,Skipper of light blue motor boat phoned coastg...,Fishing Vessel,Commercial,Commercial fishing,Injured/ill,Vessel
19664,368792,2022-08-11 15:50:00.000,Dinghy capsized east of Spit bank launched wes...,Sailing Dinghy,Sailing,Sailing dinghy,Capsize/swamping,Small Craft


# ACC

In [3]:
df_data = df_main[['IncidentId', 'Narrative', 'ACC']]
df_data

,IncidentId,Narrative,ACC
0,1014,The station mechanic was asked by harbour staf...,Commercial fishing
1,1020,Launched on service to a broken down dinghy to...,Motorboat
2,1037,Was called by a member of the public to a repo...,Rowing boat
3,1040,WHILST COMING BACK DOWN MORTLAKE REACH WE NOTI...,Motorboat
4,1048,Launched to 10M fishing vessel Likely Lad WH32...,Commercial fishing
...,...,...,...
19661,368779,"13 Metre sailing boat, 1.4 metre draught. has ...",Sailing vessel
19662,368786,Launched to a report of a boat broken down nea...,Motorboat
19663,368788,Skipper of light blue motor boat phoned coastg...,Commercial fishing
19664,368792,Dinghy capsized east of Spit bank launched wes...,Sailing dinghy


In [4]:
df_data['category_id'] = df_data.ACC.factorize()[0]
df_data = df_data[['Narrative', 'ACC', 'category_id']].drop_duplicates()

C:\Users\Joe_Davies\AppData\Local\Temp\ipykernel_11528\2820193000.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_data['category_id'] = df_data.ACC.factorize()[0]


In [5]:
df_data = df_data.dropna()

In [6]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=5, ngram_range=(1, 3), stop_words='english')

In [7]:
features = tfidf.fit_transform(df_data.Narrative).toarray()
labels = df_data.category_id

In [12]:
print(np.mean(cross_val_score(model, features, labels, scoring='accuracy')))

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\model_selection\_split.py:676: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


0.8036827545357113


In [8]:
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [15]:
model = LinearSVC()
model.fit(x_train, y_train)

LinearSVC()

In [17]:
predictions = model.predict(x_test)

In [22]:
pd.DataFrame(cm(y_test, predictions))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,366,68,0,4,0,0,0,0,12,0,1,0,1,0,0,0,0,0
1,50,1031,4,16,2,9,0,6,71,3,4,0,9,0,0,0,0,0
2,4,24,29,0,0,0,0,2,3,1,0,0,4,0,0,0,0,0
3,7,35,0,42,0,0,0,0,7,0,0,0,0,0,0,0,0,0
4,3,5,0,0,179,1,0,1,0,1,0,0,0,0,0,2,1,0
5,0,11,0,1,1,122,0,0,0,0,0,0,0,0,0,0,0,0
6,1,3,0,0,1,0,15,0,1,0,0,0,1,1,0,1,0,0
7,2,11,0,0,1,0,0,49,22,1,0,0,3,0,0,0,0,0
8,14,80,0,3,1,2,1,4,951,3,0,0,3,1,0,0,0,0
9,2,15,2,3,2,0,0,2,4,4,0,0,8,0,0,1,0,0


In [28]:
model.score(x_test, y_test)

0.8115154807170016

## Parameter Testing

In [33]:
iterations = [100, 250, 500, 1000, 2000, 3000]
scores = {}
for i in iterations:
    model = LinearSVC(max_iter = i)
    model.fit(x_train, y_train)
    scores[i] = model.score(x_test, y_test)

In [34]:
scores

{100: 0.8115154807170016,
 250: 0.8115154807170016,
 500: 0.8115154807170016,
 1000: 0.8115154807170016,
 2000: 0.8115154807170016,
 3000: 0.8115154807170016}

In [35]:
ledom = LinearSVC(max_iter = 1)
ledom.fit(x_train, y_train)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC(max_iter=1)

In [36]:
ledom.score(x_test, y_test)

0.8030961434003259

In [37]:
smol_iters = [1, 5, 10, 25, 50, 100]
scores = {}
for i in smol_iters:
    model = LinearSVC(max_iter = i)
    model.fit(x_train, y_train)
    scores[i] = model.score(x_test, y_test)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [38]:
scores

{1: 0.7577403585008148,
 5: 0.8136882129277566,
 10: 0.8109722976643129,
 25: 0.8115154807170016,
 50: 0.8115154807170016,
 100: 0.8115154807170016}

Max number of iterations can be reduced from 1000 to 25 with no loss at all.

Let's test out some more vectorizer options

In [6]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=6, ngram_range=(1, 3), stop_words='english')

In [7]:
features = tfidf.fit_transform(df_data.Narrative).toarray()
labels = df_data.category_id

In [8]:
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [9]:
model = LinearSVC(max_iter = 25)
model.fit(x_train, y_train)
model.score(x_test, y_test)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.8126018468223791

In [11]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=7, ngram_range=(1, 3), stop_words='english')

In [12]:
features = tfidf.fit_transform(df_data.Narrative).toarray()

In [14]:
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [15]:
model = LinearSVC(max_iter = 25)
model.fit(x_train, y_train)
model.score(x_test, y_test)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.8112438891906573

In [16]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=4, ngram_range=(1, 3), stop_words='english')

In [17]:
features = tfidf.fit_transform(df_data.Narrative).toarray()
labels = df_data.category_id

x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

model = LinearSVC(max_iter = 25)
model.fit(x_train, y_train)
model.score(x_test, y_test)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.8161325366648561

In [8]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=3, ngram_range=(1, 3), stop_words='english')

In [9]:
features = tfidf.fit_transform(df_data.Narrative).toarray()

In [10]:
labels = df_data.category_id

x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

model = LinearSVC(max_iter = 25)
model.fit(x_train, y_train)
model.score(x_test, y_test)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.815317762085823

min_df = 4 is best

In [6]:
tfidf = TfidfVectorizer(sublinear_tf=False, min_df=4, ngram_range=(1, 3), stop_words='english')

In [7]:
features = tfidf.fit_transform(df_data.Narrative).toarray()

In [8]:
labels = df_data.category_id

x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

model = LinearSVC(max_iter = 25)
model.fit(x_train, y_train)
model.score(x_test, y_test)

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


0.8115154807170016

Keep sublinear on

# Final Model

In [14]:
def model_creation(fname, category):
    df = pd.read_csv(fname)[['Narrative', category]]
    df['category_id'] = df[category].factorize()[0]
    df = df[['Narrative', category, 'category_id']].drop_duplicates()
    df = df.dropna()
    tfidf = TfidfVectorizer(sublinear_tf=False, min_df=4, ngram_range=(1, 3), stop_words='english')
    labels = df.category_id
    features = tfidf.fit_transform(df.Narrative).toarray()
    x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)
    model = LinearSVC(max_iter = 25)
    model.fit(x_train, y_train)
    print('Model Accuracy: %d'%(round(model.score(x_test, y_test), 4)*100))
    
    cat = category.lower().replace(' ', '_')
    name = 'model_%s.pkl'%(cat)
    pkl.dump(model, open(name, 'wb'))
    
    return model

In [15]:
model = model_creation('../data/narrative_craft.csv', 'ACC')

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model Accuracy: 81


In [16]:
model = model_creation('../data/big_sick_narrative.csv', 'Was Big Sick')

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model Accuracy: 95


In [17]:
model = model_creation('../data/narrative_craft.csv', 'Craft Type')

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model Accuracy: 72


# Making this into a production model
We are not going to be able to have a "one-size-fits-all" type model. The only way this would be possible is to take every column we wish to take out, convert every category into a numerical value, put all of those into one target column and train the model. Instead, we create a function that simply makes and trains a model if none exists, saving it as a pickle file. If the model does exist, we use it.

In [3]:
from pathlib import Path

In [4]:
def model_chooser(fname, category):
    lower_category = category.lower().replace(' ', '_')
    model_name = 'model_%s.pkl'%(lower_category)
    model_directory = 'C:/Users/Joe_Davies/Documents/Analysis/models/'
    path = model_directory+model_name 
    if Path(path).is_file():
        print('Model Exists')
        model = pkl.load(open(path, 'rb'))
        return model
    else:
        print("Model Doesn't Exist: Creating...")
        model = model_creation(fname, category)
        return model

In [16]:
model_chooser('', 'Craft Type')

Model Exists


LinearSVC(max_iter=25)

In [17]:
model_chooser('../data/big_sick_narrative.csv', 'Was Big Sick')

Model Doesn't Exist: Creating...


C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model Accuracy: 95


LinearSVC(max_iter=25)

# Stuff for Roger

In [36]:
df = pd.read_csv('../data/DataForJoe_short.csv', encoding='cp1252')
df

,AIC_Group,AIC,AIC_Subtype,Narrative
0,Miscellaneous,Unknown,Unknown,Tower LB was tasked to a report of a person TT...
1,Miscellaneous,Unknown,Unknown,The lifeboat was launched to a report of a mis...
2,People,Person ashore,On shore,We were requested to launch to a female that w...
3,Miscellaneous,Unknown,Unknown,Tower LB was tasked to a report of a male in t...
4,Miscellaneous,Other,Other,"Comprehensive search of Firestone Bay, Barne P..."
...,...,...,...,...
17218,People,Person ashore,On shore,Launched by Coastguard to report of female thr...
17219,People,Person in water,In water,Launched to report of person in water in River...
17220,Miscellaneous,Other,Other,Both boats requested to launch to search for t...
17221,People,Person ashore,On shore,Adult Male threatening to enter water at Shark...


In [38]:
model_df = df.loc[(df.AIC != 'Unknown') & (df.AIC != 'Unknown craft') & (df.AIC != 'Unknown person')]
model_df

,AIC_Group,AIC,AIC_Subtype,Narrative
2,People,Person ashore,On shore,We were requested to launch to a female that w...
4,Miscellaneous,Other,Other,"Comprehensive search of Firestone Bay, Barne P..."
26,Miscellaneous,Other,Other,Poole Lifeboats were both tasked this morning ...
28,People,Other,Injured/ill,"Female at Greenwich Pier that hit her head, st..."
34,People,Person on craft,Unknown person,Buckie lifeboat was launched with Macduff ILB ...
...,...,...,...,...
17218,People,Person ashore,On shore,Launched by Coastguard to report of female thr...
17219,People,Person in water,In water,Launched to report of person in water in River...
17220,Miscellaneous,Other,Other,Both boats requested to launch to search for t...
17221,People,Person ashore,On shore,Adult Male threatening to enter water at Shark...


In [87]:
unknown_df = df[(df.AIC == 'Unknown') | (df.AIC == 'Unknown craft') | (df.AIC == 'Unknown person')]
unknown_df

,AIC_Group,AIC,AIC_Subtype,Narrative
0,Miscellaneous,Unknown,Unknown,Tower LB was tasked to a report of a person TT...
1,Miscellaneous,Unknown,Unknown,The lifeboat was launched to a report of a mis...
3,Miscellaneous,Unknown,Unknown,Tower LB was tasked to a report of a male in t...
5,Miscellaneous,Unknown,Unknown,"Comprehensive search of Firestone Bay, Barne P..."
6,People,Unknown person,Missing/overdue,ILB and BB were requested to search the River ...
...,...,...,...,...
17049,People,Unknown person,Other,"On 4/7/21 at 04:45, IRH, H-001, was launched ..."
17112,People,Unknown person,Other,On 18/7/21 at 13:21 IRH was diverted from LBI0...
17115,People,Unknown person,Other,"On 18/7/21 at 13:46, IRH was diverted from LBI..."
17183,People,Unknown person,Trapped/stuck,"On 12/10/21 at 12:55, IRH (H-004) was launched..."


df = pd.read_csv(fname)[['Narrative', category]]
    df['category_id'] = df[category].factorize()[0]
    df = df[['Narrative', category, 'category_id']].drop_duplicates()
    df = df.dropna()
    tfidf = TfidfVectorizer(sublinear_tf=False, min_df=4, ngram_range=(1, 3), stop_words='english')
    labels = df.category_id
    features = tfidf.fit_transform(df.Narrative).toarray()
    x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)
    model = LinearSVC(max_iter = 25)
    model.fit(x_train, y_train)
    print('Model Accuracy: %d'%(round(model.score(x_test, y_test), 4)*100))
    
    cat = category.lower().replace(' ', '_')
    name = 'model_%s.pkl'%(cat)
    pkl.dump(model, open(name, 'wb'))
    
    return model

In [95]:
tfidf = TfidfVectorizer(sublinear_tf=False, min_df=4, ngram_range=(1, 3), stop_words='english')

In [67]:
data = model_df[['AIC', 'Narrative']]

In [68]:
data['category_id'] = data.AIC.factorize()[0]

C:\Users\Joe_Davies\AppData\Local\Temp\ipykernel_14112\1990397570.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['category_id'] = data.AIC.factorize()[0]


In [69]:
data = data.drop_duplicates()
data = data.dropna()

In [70]:
labels = data.category_id

In [98]:
features = tfidf.fit_transform(data.Narrative).toarray()

In [72]:
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)
model = LinearSVC(max_iter = 25)
model.fit(x_train, y_train)
print('Model Accuracy: %d'%(round(model.score(x_test, y_test), 4)*100))

C:\Users\Joe_Davies\Anaconda3\lib\site-packages\sklearn\svm\_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model Accuracy: 69


In [99]:
unknown_df = unknown_df.dropna()

In [100]:
unknown_features = tfidf.transform(unknown_df.Narrative)

In [125]:
preds = model.predict(unknown_features)

In [107]:
category_map = dict(zip(labels, data.AIC))

In [126]:
results = [category_map[i] for i in preds]

In [130]:
unknown_df['predictions'] = results
unknown_df = unknown_df.drop(columns=['AIC_Group', 'AIC_Subtype'])
unknown_df

,AIC,Narrative,predictions
0,Unknown,Tower LB was tasked to a report of a person TT...,Person ashore
1,Unknown,The lifeboat was launched to a report of a mis...,Person ashore
3,Unknown,Tower LB was tasked to a report of a male in t...,Person in water
5,Unknown,"Comprehensive search of Firestone Bay, Barne P...",Other
6,Unknown person,ILB and BB were requested to search the River ...,Person ashore
...,...,...,...
17049,Unknown person,"On 4/7/21 at 04:45, IRH, H-001, was launched ...",Person in water
17112,Unknown person,On 18/7/21 at 13:21 IRH was diverted from LBI0...,Person in water
17115,Unknown person,"On 18/7/21 at 13:46, IRH was diverted from LBI...",Person in water
17183,Unknown person,"On 12/10/21 at 12:55, IRH (H-004) was launched...",Person in water


In [131]:
unknown_df.to_csv('../data/roger_predictions.csv')